## PageIndex – Document Indexing Architecture

```mermaid
flowchart TD
    A["RAW PDF DOCUMENT<br/>Any long structured document"]

    B["TOC DETECTION<br/>Scan first N pages for existing headings"]

    A --> B

    B --> C{"Does the document<br/>have a TOC?"}

    C -->|YES| D["PARSE EXISTING TOC<br/>Extract chapter structure"]
    C -->|NO| E["LLM READS PAGES<br/>Infer headings and document structure"]

    D --> F["SECTION-AWARE SPLITTING"]
    E --> F

    F --> F1["Respect logical boundaries<br/>Split by chapters, sections and subsections<br/>NOT arbitrary token counts"]

    F1 --> G["LLM SUMMARIZES EACH SECTION"]

    G --> G1["Generate node metadata<br/>Node ID<br/>Title<br/>Page number / Page range<br/>Summary"]

    G1 --> H["ASSEMBLE HIERARCHICAL TREE"]

    H --> H1["Parent Nodes"]
    H --> H2["Child Nodes"]
    H2 --> H3["Grandchild Nodes"]

    subgraph JSON["OUTPUT: IN-CONTEXT JSON TREE"]
        direction TB

        I["FINANCIAL STABILITY<br/>node_id: 0006<br/>page: 21<br/>summary: Overview of financial stability"]

        J["MONITORING VULNERABILITIES<br/>node_id: 0007<br/>pages: 22–25"]

        K["INTERNATIONAL COOPERATION<br/>node_id: 0008<br/>pages: 28–31"]

        I --> J
        I --> K
    end

    H1 --> I

    classDef document fill:#E8F1FA,stroke:#3977A8,color:#123B5D,stroke-width:2px
    classDef process fill:#EDEAF8,stroke:#8174B5,color:#39305D,stroke-width:1.5px
    classDef branch fill:#DDF2E9,stroke:#32866B,color:#15513D,stroke-width:1.5px
    classDef tree fill:#DDF2E9,stroke:#32866B,color:#15513D,stroke-width:1.5px
    classDef output fill:#F2F0E7,stroke:#8B856B,color:#383729,stroke-width:1.5px

    class A document
    class B,F,F1,G,G1 process
    class D,E,H branch
    class H1,H2,H3 tree
    class I,J,K output
```

## Simplified hierarchy of the generated JSON tree
```mermaid
flowchart TD
    A["Root Node<br/>Entire PDF Document"]

    A --> B["Chapter 1"]
    A --> C["Chapter 2"]
    A --> D["Chapter 3"]

    B --> B1["Section 1.1"]
    B --> B2["Section 1.2"]

    B1 --> B11["Subsection 1.1.1"]
    B1 --> B12["Subsection 1.1.2"]

    C --> C1["Section 2.1"]
    C --> C2["Section 2.2"]

    D --> D1["Section 3.1"]

    classDef root fill:#DCEAF7,stroke:#3977A8,color:#123B5D,stroke-width:2px
    classDef parent fill:#DDF2E9,stroke:#32866B,color:#15513D
    classDef child fill:#F2F0E7,stroke:#8B856B,color:#383729

    class A root
    class B,C,D parent
    class B1,B2,C1,C2,D1,B11,B12 child
```

# 📑 PageIndex — Vectorless RAG Crash Course
### Reasoning-based RAG with No Vector DB, No Chunking


## 🧠 What You'll Learn

| # | Topic |
|---|-------|
| 1 | Why Vector RAG fails on professional documents |
| 2 | How PageIndex builds a tree index from a PDF |
| 3 | LLM Tree Search — reasoning over structure |
| 4 | Full end-to-end Vectorless RAG pipeline |
| 5 | Expert-guided retrieval (domain knowledge injection) |
| 6 | Chat API — zero LLM setup |
| 7 | Self-hosted open-source option |

---

## 🔑 Key Concept

> **Traditional RAG** → chunk → embed → cosine similarity → retrieve  
> **PageIndex RAG** → build tree → LLM reasons over tree → retrieve exact sections

**The problem with vector RAG:**  
`Similarity ≠ Relevance`  
A chunk about "market conditions" may score higher than the actual answer section just because it shares more words with your query.


---
## 📦 Section 1: Install & Setup

**What we do here:**
- Install PageIndex SDK + OpenAI
- Load API keys from `.env`
- Initialize both clients

> 🔑 Get your **PageIndex API key** from: https://dash.pageindex.ai/api-keys  
> 🔑 Get your **OpenAI API key** from: https://platform.openai.com


In [6]:
# ── Create a .env file (run this once) ──────────────────────────────────────
# Uncomment and fill in your keys, then run this cell ONCE

# env_content = """
# PAGEINDEX_API_KEY=your_pageindex_key_here
# OPENAI_API_KEY=your_openai_key_here
# """
# with open(".env", "w") as f:
#     f.write(env_content.strip())
# print("✅ .env file created")

In [7]:
import os, json, time
from dotenv import load_dotenv

load_dotenv()

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
GROQ_API_KEY    = os.getenv("GROQ_API_KEY")

print("PageIndex key loaded:", "✅" if PAGEINDEX_API_KEY else "❌ Missing!")
print("Groq OpenAI key loaded:", "✅" if GROQ_API_KEY    else "❌ Missing!")

PageIndex key loaded: ✅
Groq OpenAI key loaded: ✅


In [8]:
from pageindex import PageIndexClient
from groq import Groq

pi_client     = PageIndexClient(api_key=PAGEINDEX_API_KEY)
openai_client = Groq(api_key=GROQ_API_KEY)

print("✅ PageIndex client ready")
print("✅ Groq OpenAI client ready")

✅ PageIndex client ready
✅ Groq OpenAI client ready


---
## 🌲 Section 2: Upload & Index a PDF

**What happens here:**
1. Upload your PDF to the PageIndex cloud
2. PageIndex uses an LLM to read the document structure
3. Builds a hierarchical **tree index** (like a smart Table of Contents)
4. Returns a `doc_id` for all future operations

**Why NO chunking?**  
Instead of cutting the document into arbitrary 500-token pieces, PageIndex respects the document's natural section boundaries — chapters, sub-sections, paragraphs — as the author intended.


In [9]:
# ── Upload your PDF ─────────────────────────────────────────────────────────
# Replace with the path to your PDF file
# Great candidates: Annual reports, research papers, legal docs, textbooks

PDF_PATH = "./sample_document.pdf"   # ← change this

print(f"📤 Uploading: {PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

print(f"✅ Uploaded!")
print(f"📋 Document ID: {doc_id}")
print("   (Save this ID — you'll use it throughout the notebook)")

📤 Uploading: ./sample_document.pdf
✅ Uploaded!
📋 Document ID: pi-cmvy66yt700kn0bnyq74otm1k
   (Save this ID — you'll use it throughout the notebook)


C:\Users\45ary\AppData\Local\Temp\ipykernel_16880\2826927943.py:8: UserWarning: Document "sample_document.pdf" was stored as "sample_document_2.pdf".
  result = pi_client.submit_document(PDF_PATH)


In [10]:
# ── Poll until processing is complete ───────────────────────────────────────
# PageIndex builds the tree asynchronously.
# For a 50-page PDF this typically takes 30–90 seconds.

print("⏳ Building tree index...")
print("   (This runs once per document — the index is cached for reuse)")

while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"   Status: {status}")
    
    if status == "completed":
        print("\n✅ Tree index ready!")
        break
    elif status == "failed":
        print("\n❌ Processing failed. Check your PDF format.")
        break
    
    time.sleep(5)

⏳ Building tree index...
   (This runs once per document — the index is cached for reuse)
   Status: queued
   Status: processing
   Status: processing
   Status: processing
   Status: processing
   Status: processing
   Status: processing
   Status: processing
   Status: processing
   Status: completed

✅ Tree index ready!


---
## 🔍 Section 3: Inspect the Tree Structure

**What the tree looks like:**

```
Document
├── Introduction (pages 1-3)
│   └── Background (pages 1-2)
├── Financial Stability (pages 21-31)
│   ├── Monitoring Vulnerabilities (pages 22-28)
│   └── International Cooperation (pages 28-31)
└── Conclusion (pages 45-47)
```

Each node has:
- `node_id` — unique ID used during retrieval
- `title` — section heading
- `page_index` — page number in original PDF
- `text` — section summary (when `node_summary=True`)
- `nodes` — child sections (nested)

**This structure is what the LLM reasons over during retrieval.**


In [11]:
# ── Fetch the full tree ─────────────────────────────────────────────────────
tree_result  = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])

print(f"📊 Top-level sections: {len(pageindex_tree)}")
print("\n🌲 Raw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

📊 Top-level sections: 1

🌲 Raw tree (first node):
{
  "title": "Data Communication & Computer Networks",
  "node_id": "0000",
  "start_index": 1,
  "end_index": 35,
  "summary": "This document provides a comprehensive exam-ready reference for Data Communication and Computer Networks, structured across 72 sections. It covers fundamental communication concepts, system characteristics (delivery, accuracy, timeliness, and jitter), and data flow types (simplex, half-duplex, and full-duplex). It outlines network introductions, components (end and intermediary devices, media, protocols), evaluation criteria, layered architecture benefits, and communication standards/organizations. \n\nThe text thoroughly details both the 7-layer OSI reference model and the 4-layer TCP/IP model, breaking down the functions, protocols, and Protocol Data Units (PDUs) associated with each layer. It examines guided transmission media (twisted pair, coaxial, fiber optics) and unguided media (radio waves, microwaves

In [12]:
# ── Pretty-print the full tree ───────────────────────────────────────────────
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("📚 Full Document Structure:\n")
print_tree(pageindex_tree)

📚 Full Document Structure:

[0000] Data Communication & Computer Networks  (p.?)
  └─ [0001] Data Communication & Computer Networks (intro)  (p.?)
  └─ [0002] 1. Data Communication Concepts  (p.?)
  └─ [0003] 2. Characteristics of Data Communication  (p.?)
  └─ [0004] 3. Data Flow  (p.?)
  └─ [0005] 4. Introduction to Networks  (p.?)
  └─ [0006] 5. Network Components  (p.?)
  └─ [0007] 6. Network Criteria  (p.?)
  └─ [0008] 7. Introduction to Layered Architecture  (p.?)
  └─ [0009] 8. Protocols and Standards  (p.?)
  └─ [0010] 9. Standards  (p.?)
  └─ [0011] 10. Standardization Organizations  (p.?)
  └─ [0012] 11. Reference Models  (p.?)
  └─ [0013] 12. OSI Reference Model  (p.?)
  └─ [0014] 13. Physical Layer  (p.?)
  └─ [0015] 14. Data Link Layer  (p.?)
  └─ [0016] 15. Network Layer  (p.?)
  └─ [0017] 16. Transport Layer  (p.?)
  └─ [0018] 17. Session Layer  (p.?)
  └─ [0019] 18. Presentation Layer  (p.?)
  └─ [0020] 19. Application Layer  (p.?)
  └─ [0021] 20. OSI Layer Summary  (p.

In [13]:
# ── Count total nodes ────────────────────────────────────────────────────────
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"🔢 Total nodes in tree: {total}")
print("   Each node = one retrievable section of the document")

🔢 Total nodes in tree: 74
   Each node = one retrievable section of the document


---
## 🧠 Section 4: LLM Tree Search — The Core of PageIndex

**This is where PageIndex fundamentally differs from vector RAG.**

### Vector RAG retrieval:
```
query → embed → cosine_similarity(query_vec, all_chunk_vecs) → top-k chunks
```
*Problem: finds what's similar, not what's relevant*

### PageIndex retrieval:
```
query + tree → LLM reasons → "node 0007 and 0008 contain the answer"
```
*Advantage: LLM understands document structure, context, and intent*

**The LLM acts like a human expert scanning a Table of Contents.**


In [14]:
# ── LLM Tree Search Function ─────────────────────────────────────────────────

def llm_tree_search(query: str, tree: list, model: str = "openai/gpt-oss-120b") -> dict:
    """
    Core PageIndex retrieval:
    Sends the query + document tree to an LLM.
    LLM reasons over the structure and returns relevant node_ids.
    
    Returns: dict with 'thinking' (reasoning) and 'node_list' (node IDs)
    """
    
    # Compress tree to save tokens — only send titles + short summaries
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]  # first 150 chars
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out
    
    compressed_tree = compress(tree)
    
    prompt = f"""You are given a query and a document's tree structure (like a Table of Contents).
Your task: identify which node IDs most likely contain the answer to the query.
Think step-by-step about which sections are relevant.

Query: {query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<your step-by-step reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    
    return json.loads(response.choices[0].message.content)

In [15]:
# ── Test with a sample query ─────────────────────────────────────────────────
query = "What is the syllabus covered in Data Link Layer?"

print(f"🔍 Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)

print("🧠 LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("🎯 Selected Node IDs:", result.get("node_list", []))

🔍 Query: What is the syllabus covered in Data Link Layer?

🧠 LLM Reasoning:
The query asks for the syllabus covered in the Data Link Layer. The document tree contains a dedicated section for the Data Link Layer with node_id "0015" titled "14. Data Link Layer". This node is the primary source for the definition and topics (syllabus) of the Data Link Layer. While other nodes (e.g., bridges, switches) are related, the syllabus itself is most directly described in node 0015. Therefore, the most relevant node is 0015.

🎯 Selected Node IDs: ['0015']


---
## ⚙️ Section 5: Full End-to-End RAG Pipeline

**3 steps:**
1. **Tree Search** → LLM picks relevant `node_ids`
2. **Retrieve** → Fetch the actual section content from those nodes  
3. **Generate** → LLM writes a grounded answer with page citations

**What makes this better than vector RAG:**
- Retrieved content has titles + page numbers (traceable)
- LLM can cite exactly *which section* the answer comes from
- No hallucination from irrelevant chunks


In [16]:
# ── Helper: Find nodes by ID ─────────────────────────────────────────────────

def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively walk the tree and collect nodes matching target_ids."""
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

In [17]:
# ── Generate answer from retrieved nodes ─────────────────────────────────────

def generate_answer(query: str, nodes: list, model: str = "openai/gpt-oss-120b") -> str:
    """
    Takes retrieved nodes as context and generates a grounded answer.
    Instructs the LLM to cite section titles and page numbers.
    """
    if not nodes:
        return "⚠️ No relevant sections found in the document."
    
    # Build context string from retrieved nodes
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
    context = "\n\n---\n\n".join(context_parts)
    
    prompt = f"""You are an expert document analyst.
Answer the question using ONLY the provided context.
For every claim you make, cite the section title and page number in parentheses.
Be concise and precise.

Question: {query}

Context:
{context}

Answer:"""
    
    response = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

In [18]:
# ── The complete Vectorless RAG function ─────────────────────────────────────

def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:
    
    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")
    
    # Step 1: Tree Search
    search_result  = llm_tree_search(query, tree)
    node_ids       = search_result.get("node_list", [])
    
    if verbose:
        print(f"\n🧠 Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"🎯 Retrieved node IDs: {node_ids}")
    
    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)
    
    if verbose:
        print(f"📄 Sections found: {[n['title'] for n in nodes]}")
    
    # Step 3: Generate answer
    answer = generate_answer(query, nodes)
    
    if verbose:
        print(f"\n📝 Answer:\n{answer}")
    
    return answer

In [19]:
# ── Run the full pipeline ────────────────────────────────────────────────────
answer = vectorless_rag(
    query="What are the syllabus covered in data link layer?",
    tree=pageindex_tree
)

🔍 Query: What are the syllabus covered in data link layer?

🧠 Reasoning: The query asks for the syllabus/topics covered in the Data Link Layer. The document tree includes a specific section titled '14. Data Link Layer' (node_id 0015) which likely outlines its definition, f...
🎯 Retrieved node IDs: ['0015']
📄 Sections found: ['14. Data Link Layer']

📝 Answer:
The syllabus for the Data Link Layer includes:

- **Definition** – node‑to‑node delivery of data and conversion of network‑layer packets into frames【14. Data Link Layer, p.?】  
- **Functions** –  
  • Framing (segmenting data into frames)【14. Data Link Layer, p.?】  
  • Physical Addressing (using MAC addresses)【14. Data Link Layer, p.?】  
  • Error Detection (identifying corrupted frames)【14. Data Link Layer, p.?】  
  • Flow Control (regulating transmission rate)【14. Data Link Layer, p.?】  
  • Access Control (deciding which device may use a shared medium)【14. Data Link Layer, p.?】  
- **Devices** – Switches, bridges (PDU: Frame)【1

In [20]:
# ── Test with multiple queries ───────────────────────────────────────────────
test_queries = [
    "What are the syllabus covered in physical layer?",
    "What are the syllabus covered in OSI Model?",
    "Summarize the syllabus of data link layer?",
]

for q in test_queries:
    print()
    ans = vectorless_rag(q, pageindex_tree, verbose=False)
    print(f"Q: {q}")
    print(f"A: {ans[:300]}...")
    print("-" * 55)


Q: What are the syllabus covered in physical layer?
A: The physical‑layer syllabus includes the following topics:

* **Definition of the Physical Layer** – the lowest OSI layer that transmits raw bits over a medium (13. Physical Layer, page ?).

* **Functions**
  - **Bit Transmission** – converting bits to signals for transmission (13.2.1, page ?).
  - ...
-------------------------------------------------------

Q: What are the syllabus covered in OSI Model?
A: The OSI model syllabus consists of the seven layered topics below (Section “12. OSI Reference Model”, Page ?):

| Layer (number) | Layer name | Core focus (as outlined in the corresponding layer sections) |
|---|---|---|
| 7 | **Application** – network services to user applications (Section “19. App...
-------------------------------------------------------

Q: Summarize the syllabus of data link layer?
A: **Data Link Layer Syllabus**

- **Definition:** Provides node‑to‑node delivery, converting network‑layer packets into frame

---
## 🎓 Section 6: Expert-Guided Retrieval

**The killer feature no one talks about.**

With vector RAG, injecting domain expertise requires **fine-tuning the embedding model** — expensive and time-consuming.

With PageIndex, you just **add rules to the prompt**:

```
"If the query mentions EBITDA → prioritize the MD&A section"
"If the query is about risks  → check Part I, Item 1A"
```

This makes PageIndex instantly adaptable to any domain — finance, legal, medical, technical — without any model training.


In [21]:
# ── Define domain expert rules ───────────────────────────────────────────────
# These are routing rules that tell the LLM WHERE to look for specific queries.
# Think of it as encoding a senior analyst's institutional knowledge.


In [22]:

FINANCIAL_EXPERT_RULES = """
Expert routing rules for financial documents (10-K, annual reports):
- EBITDA, profitability queries    → MD&A section (Management Discussion & Analysis)
- Liquidity, cash flow queries     → Cash Flow Statement + liquidity footnotes
- Risk factor queries              → Part I, Item 1A (Risk Factors)  
- Revenue breakdown queries        → Segment reporting or Item 7
- Forward-looking / strategy       → CEO letter, Outlook, Strategy section
- Debt, credit, leverage queries   → Balance Sheet + debt footnotes
- Regulatory / compliance queries  → Legal Proceedings or regulatory filings
"""

print("✅ Expert rules defined")
print("   These get injected into the retrieval prompt at query time.")

✅ Expert rules defined
   These get injected into the retrieval prompt at query time.


In [23]:
# ── Expert Routing Rules — Advanced Route of Learning AI ─────────────────────
# Krish Naik Academy | 21 Modules | 38 Sections | 481 Topics
FINANCIAL_EXPERT_RULES = """
Route queries to the correct module using these rules:
 
M1  Neural Network Refresher   → backprop, activations, optimizers, PyTorch basics
M2  Hardware                   → GPU, TPU, Apple Silicon, compute infrastructure
M3  Transformers 101           → attention, self-attention, encoder-decoder, MHA
M4  Tokenization               → BPE, WordPiece, SentencePiece, Byte Latent Transformers
M5  Finetuning Architectures   → hands-on BERT/GPT/T5 finetuning, Hugging Face
M6  KV Cache & Attention       → KV cache, Flash Attention, MQA, GQA, RoPE, vLLM
M7  Scaling Laws               → Kaplan, Chinchilla, compute-optimal training
M8  Mixture of Experts         → MoE, sparse computation, Mixture of Depths
M9  Modern LLM Finetuning      → LoRA, QLoRA, SFT, DPO, PPO, RLHF, GRPO, ORPO,
                                  quantization, TRL, Unsloth, synthetic data,
                                  reasoning models, evaluation, deployment
M10 SLM                        → small language models, pruning, when SLM vs LLM
M11 Knowledge Distillation     → student-teacher, soft labels, DistilBERT, DeepSeek-R1
M12 Hybrid Architectures       → Mamba, RWKV, SSMs, Jamba, Nemotron, beyond Transformers
M13 Vision Foundations         → ViT, patch embeddings, CLIP, SigLIP, DINOv2
M14 Visual Language Models     → VLM architecture, aligner, multimodal reasoning
M15 Stable Diffusion & DiT     → DDPM, latent diffusion, FLUX.1, ControlNet, DreamBooth
M16 Embedding Models           → dense, sparse, binary, Matryoshka, MRL, fine-tuning
M17 RAG                        → chunking, BM25, ColBERT, hybrid RAG, rerankers,
                                  self/corrective/adaptive/agentic RAG, Graph RAG,
                                  multi-modal RAG, ColPali, RAG security
M18 Context Engineering        → prompt vs context engineering, memory architecture,
                                  context compression, KV cache, agent context lifecycle
M19 DSPy                       → signatures, modules, MIPROv2, self-optimizing RAG
M20 Agents                     → ReAct, MCP, LangGraph, CrewAI, browser agents,
                                  A2A, guardrails, observability, evaluation
M21 RL                         → PPO, GRPO, DAPO, GSPO, CISPO, reward models,
                                  RLHF vs RLVR, policy gradient, DeepSeek-R1 training
 
Cross-cutting rules:
- "learning path / where to start"     → M1 → M2 → M3 in order
- "production / deployment / serving"  → M9 (quantization) + M20 (agents)
- "fine-tuning vs RAG"                 → M9 + M17 + M18
- "multimodal / vision + language"     → M13 + M14 + M17 (multi-modal RAG)
- "reasoning models / test-time RL"    → M9 (reasoning) + M21 (GRPO/DAPO)
"""

In [24]:
# ── Expert-guided tree search ────────────────────────────────────────────────

def llm_tree_search_with_expert(
    query: str,
    tree: list,
    expert_rules: str,
    model: str = "openai/gpt-oss-120b"
) -> dict:
    """
    Same as llm_tree_search() but with domain expert rules injected.
    The LLM uses these rules to guide its reasoning.
    """
    
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {"node_id": n["node_id"], "title": n["title"],
                     "page": n.get("page_index", "?"),
                     "summary": n.get("text", "")[:150]}
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out

    prompt = f"""You are a domain expert analyzing a document.
Find all node IDs that most likely contain the answer to the query.
Use the expert routing rules below to guide your reasoning.

Query: {query}

Document Tree:
{json.dumps(compress(tree), indent=2)}

Expert Routing Rules (follow these carefully):
{expert_rules}

Reply ONLY in this JSON format:
{{
  "thinking": "<your reasoning, referencing the expert rules>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

In [25]:
# ── Test expert-guided retrieval ─────────────────────────────────────────────
query = "Details of the modern llm finetuning?"

print(f"🔍 Query: {query}\n")

# Without expert rules
print("── Without Expert Rules ──")
basic   = llm_tree_search(query, pageindex_tree)
print("Nodes:", basic.get("node_list"))

print()

# With expert rules
print("── With Expert Rules ──")
guided  = llm_tree_search_with_expert(query, pageindex_tree, FINANCIAL_EXPERT_RULES)
print("Nodes:", guided.get("node_list"))
print("Reasoning:", guided.get("thinking", "")[:300])

🔍 Query: Details of the modern llm finetuning?

── Without Expert Rules ──
Nodes: []

── With Expert Rules ──
Nodes: []
Reasoning: The query asks for details of modern LLM finetuning (e.g., LoRA, QLoRA, SFT, DPO, etc.). The provided document tree is entirely about Data Communication & Computer Networks and contains no sections related to LLMs, finetuning, or the relevant expert modules (M9). Therefore, none of the nodes are rel


In [26]:
# ── Full expert-guided RAG ───────────────────────────────────────────────────

def expert_rag(query: str, tree: list, rules: str) -> str:
    """Expert-guided end-to-end RAG pipeline."""
    result  = llm_tree_search_with_expert(query, tree, rules)
    nodes   = find_nodes_by_ids(tree, result.get("node_list", []))
    return generate_answer(query, nodes)

# Run it
answer = expert_rag(
    query="Details of the syllabus of modern llm finetuning",
    tree=pageindex_tree,
    rules=FINANCIAL_EXPERT_RULES
)
print(answer)

⚠️ No relevant sections found in the document.


---
## 💬 Section 7: Chat API — Zero LLM Setup

**When to use this:**
- You don't want to manage OpenAI API calls yourself
- You want a quick Q&A interface over your document
- You're building a chat product and want PageIndex to handle everything

PageIndex provides its own LLM — you just pass a question and `doc_id`.


In [27]:
# ── Single question with Chat API ────────────────────────────────────────────
# No OpenAI key needed — PageIndex runs the LLM internally

question = "What are the key findings in this document?"

response = pi_client.chat_completions(
    messages=[{"role": "user", "content": question}],
    doc_id=doc_id
)

answer = response["choices"][0]["message"]["content"]
print("💬 Chat API Answer:")
print(answer)

💬 Chat API Answer:
This document is an exam-style reference on Data Communication and Computer Networks. It has 72 sections, and the outline gives these main points. I worked from the document's section summaries and didn't read the full page text.

**Fundamentals**
- Data communication has five components: sender, receiver, message, medium and protocol.
- A good system delivers data to the right place, accurately and on time. It also keeps jitter low, meaning the delay stays consistent.
- Data flow is simplex, half-duplex or full-duplex.
- Networks are judged on performance, reliability and security.
- Protocols have three elements: syntax, semantics and timing.
- Standards are either de facto (accepted through widespread use) or de jure (formally approved). The main standards bodies are ISO, IEEE, IETF and ITU-T.

**Layered models**
- The OSI model has 7 layers: Physical, Data Link, Network, Transport, Session, Presentation and Application.
- The TCP/IP model has 4 layers: Applicatio

In [28]:
# ── Multi-turn conversation ───────────────────────────────────────────────────
# Keep the full message history for context across turns

conversation_history = []

def chat_with_doc(user_message: str, doc_id: str) -> str:
    """Chat with a document, maintaining conversation history."""
    global conversation_history
    
    conversation_history.append({"role": "user", "content": user_message})
    
    response = pi_client.chat_completions(
        messages=conversation_history,
        doc_id=doc_id
    )
    
    assistant_reply = response["choices"][0]["message"]["content"]
    conversation_history.append({"role": "assistant", "content": assistant_reply})
    
    return assistant_reply


# Simulate a 3-turn conversation
questions = [
    "What were the main revenue sources last year?",
    "How does that compare to the year before?",
    "What factors drove that change?"
]

for q in questions:
    print(f"\n👤 User: {q}")
    reply = chat_with_doc(q, doc_id)
    print(f"🤖 Assistant: {reply[:400]}...")
    print("-" * 55)


👤 User: What were the main revenue sources last year?
🤖 Assistant: I can't answer that from this document. *sample_document_2.pdf* is an exam-prep reference on Data Communication and Computer Networks. It covers the OSI and TCP/IP models, transmission media, topologies, multiplexing, switching, LANs, interconnection devices and PDU encapsulation. I checked its full outline across all 72 sections. It has no financial information, revenue figures or business-perfor...
-------------------------------------------------------

👤 User: How does that compare to the year before?
🤖 Assistant: I can't make that comparison. Neither year's revenue is available, because *sample_document_2.pdf* has no financial data.

I also searched the rest of your library. It holds only two other documents, *sample_document.pdf* and *sample_document_1.pdf*. Both are Data Communication and Computer Networks study guides, so they have no revenue figures either.

To compare year over year, I'd need a docume...
----

---
## 📊 Section 8: Vector RAG vs PageIndex — Side-by-Side

### Architecture Comparison

| Aspect | Traditional Vector RAG | PageIndex (Vectorless RAG) |
|--------|------------------------|---------------------------|
| **Document prep** | Chunk into fixed pieces | Build hierarchical tree |
| **Indexing** | Embed each chunk | LLM reads structure |
| **Storage** | Vector database | JSON file |
| **Query processing** | Embed query → ANN search | LLM reasons over tree |
| **What's retrieved** | Flat anonymous chunks | Named sections + page refs |
| **Explainability** | ❌ Opaque similarity score | ✅ Traceable reasoning |
| **Domain expertise** | ❌ Needs embedding fine-tune | ✅ Add rules to prompt |
| **Infrastructure** | Pinecone / FAISS / ChromaDB | No vector DB needed |
| **Best for** | Short, diverse documents | Long, structured documents |
| **FinanceBench accuracy** | ~80% | **98.7%** |

### When to use which

**Use Vector RAG when:**
- Documents are short and varied (FAQs, product descriptions)
- Semantic paraphrase matching is important  
- You need sub-second retrieval on millions of documents

**Use PageIndex when:**
- Documents are long and professionally structured (reports, manuals, legal docs)
- You need traceable, cited answers
- Domain expertise should guide retrieval
- You want to avoid vector DB infrastructure


In [29]:
# ── Quick comparison demo ────────────────────────────────────────────────────
# Show how the same query retrieves differently

print("=" * 55)
print("VECTOR RAG approach (conceptual):")
print("=" * 55)
print("""
query_vec = embed_model.encode("What are EBITDA risks?")
chunks    = vector_db.similarity_search(query_vec, k=5)

# Returns: 5 text fragments ranked by cosine distance
# Problem: may return "market risk" chunks, not EBITDA section
# No page numbers, no section context
""")

print("=" * 55)
print("PAGEINDEX approach (actual):")
print("=" * 55)
print("""
result = llm_tree_search("What are EBITDA risks?", tree)

# Returns: node IDs like ["0007", "0012"]
# LLM reasoning: "EBITDA is discussed in MD&A section (node 0007)
#                 and footnotes in Financial Statements (node 0012)"
# Full traceability — section title + page number
""")

VECTOR RAG approach (conceptual):

query_vec = embed_model.encode("What are EBITDA risks?")
chunks    = vector_db.similarity_search(query_vec, k=5)

# Returns: 5 text fragments ranked by cosine distance
# Problem: may return "market risk" chunks, not EBITDA section
# No page numbers, no section context

PAGEINDEX approach (actual):

result = llm_tree_search("What are EBITDA risks?", tree)

# Returns: node IDs like ["0007", "0012"]
# LLM reasoning: "EBITDA is discussed in MD&A section (node 0007)
#                 and footnotes in Financial Statements (node 0012)"
# Full traceability — section title + page number



---
## ✅ Summary

You've now built a complete **Vectorless RAG** system with PageIndex.

### What you built:

1. **`llm_tree_search()`** — LLM reasons over document tree to find relevant nodes
2. **`find_nodes_by_ids()`** — Retrieve actual section content from tree
3. **`generate_answer()`** — LLM produces cited, grounded answers
4. **`vectorless_rag()`** — Full pipeline combining all 3 steps
5. **`expert_rag()`** — Domain-guided retrieval without any fine-tuning
6. **Chat API** — Zero-setup document Q&A

### Key takeaways:

- `Similarity ≠ Relevance` — the fundamental flaw of vector search
- Tree-based reasoning gives you **traceable**, **accurate**, **explainable** retrieval
- Domain expertise injection is just **prompt engineering** — no model training needed
- 98.7% on FinanceBench vs ~80% for vector RAG

---

### 🔗 Resources
- GitHub: https://github.com/VectifyAI/PageIndex
- Docs: https://docs.pageindex.ai
- Chat Platform: https://chat.pageindex.ai
- Blog: https://pageindex.ai/blog/pageindex-intro

